#IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
import string
import nltk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All libraries loaded successfully!")

All libraries loaded successfully!


#LOAD DATA

In [3]:
df = pd.read_csv('reviews.csv')
print(f"Total reviews: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

Total reviews: 200
Columns: ['review_text', 'label']
                                         review_text  label
0  This is a one of the best apps acording to a b...      1
1  This is a pretty good version of the game for ...      1
2  this is a really cool game. there are a bunch ...      1


#PREPROCESSING

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()                                              # lowercase
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuation
    tokens = text.split()                                            # tokenize
    tokens = [t for t in tokens if t not in stop_words]             # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]              # lemmatize
    return " ".join(tokens)

df['clean_text'] = df['review_text'].apply(preprocess)
print("Preprocessing done!")
print(df[['review_text', 'clean_text']].head(3))

Preprocessing done!
                                         review_text  \
0  This is a one of the best apps acording to a b...   
1  This is a pretty good version of the game for ...   
2  this is a really cool game. there are a bunch ...   

                                          clean_text  
0  one best apps acording bunch people agree bomb...  
1  pretty good version game free lot different le...  
2  really cool game bunch level find golden egg s...  


#BUILDING VOCABULARY


In [5]:
from collections import Counter

all_words = " ".join(df['clean_text']).split()
vocab = Counter(all_words)

print(f"Vocabulary size: {len(vocab)}")
print(f"\nTop 20 most frequent words:")
for word, count in vocab.most_common(20):
    print(f"  {word}: {count}")

Vocabulary size: 1070

Top 20 most frequent words:
  game: 159
  bird: 73
  fun: 61
  get: 54
  play: 51
  angry: 51
  love: 48
  dont: 46
  app: 39
  free: 38
  time: 38
  like: 37
  kindle: 37
  level: 28
  great: 28
  really: 26
  fire: 26
  good: 24
  version: 24
  ad: 22


In [6]:
# One Hot Encoding
cv_ohe = CountVectorizer(binary=True)
X_ohe = cv_ohe.fit_transform(df['clean_text'])

print("One Hot Encoding done!")
print(f"Shape: {X_ohe.shape}")
print(f"What it means: {X_ohe.shape[0]} reviews x {X_ohe.shape[1]} unique words")

One Hot Encoding done!
Shape: (200, 1054)
What it means: 200 reviews x 1054 unique words


In [7]:
# Bag of Words
cv_bow = CountVectorizer()
X_bow = cv_bow.fit_transform(df['clean_text'])

print("Bag of Words done!")
print(f"Shape: {X_bow.shape}")
print(f"What it means: {X_bow.shape[0]} reviews x {X_bow.shape[1]} unique words")

# Show sample counts
import pandas as pd
sample = pd.DataFrame(
    X_bow[:3].toarray(),
    columns=cv_bow.get_feature_names_out()
)
print("\nFirst 3 reviews - top 10 words:")
print(sample.iloc[:, :10])

Bag of Words done!
Shape: (200, 1054)
What it means: 200 reviews x 1054 unique words

First 3 reviews - top 10 words:
   100  19quot  1st  1star  20  34ad  34killing34  4334  45  6yearold
0    0       0    0      0   0     0            0     0   0         0
1    0       0    0      0   0     0            0     0   0         0
2    0       0    0      0   0     0            0     0   0         0


In [8]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

print("TF-IDF done!")
print(f"Shape: {X_tfidf.shape}")
print(f"What it means: {X_tfidf.shape[0]} reviews x {X_tfidf.shape[1]} unique words")

# Show top important words across all reviews
feature_names = tfidf.get_feature_names_out()
tfidf_scores = X_tfidf.toarray().mean(axis=0)
top_indices = tfidf_scores.argsort()[-15:][::-1]

print("\nTop 15 most important words by TF-IDF:")
for i in top_indices:
    print(f"  {feature_names[i]}: {tfidf_scores[i]:.4f}")

TF-IDF done!
Shape: (200, 1054)
What it means: 200 reviews x 1054 unique words

Top 15 most important words by TF-IDF:
  game: 0.0775
  bird: 0.0457
  fun: 0.0455
  love: 0.0390
  get: 0.0388
  play: 0.0382
  angry: 0.0373
  dont: 0.0352
  like: 0.0315
  free: 0.0314
  app: 0.0310
  time: 0.0300
  kindle: 0.0300
  great: 0.0251
  fire: 0.0249


In [9]:
comparison = pd.DataFrame({
    'Feature': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Shape': [str(X_ohe.shape), str(X_bow.shape), str(X_tfidf.shape)],
    'What it stores': ['Word present? 0 or 1', 'Word count per review', 'Weighted importance score'],
    'Handles frequency?': ['No', 'Yes', 'Yes - penalizes common words'],
    'Best used for': ['Presence based tasks', 'Count based tasks', 'Search and key term ranking']
})

print(comparison.to_string(index=False))

         Feature       Shape            What it stores           Handles frequency?               Best used for
One Hot Encoding (200, 1054)      Word present? 0 or 1                           No        Presence based tasks
    Bag of Words (200, 1054)     Word count per review                          Yes           Count based tasks
          TF-IDF (200, 1054) Weighted importance score Yes - penalizes common words Search and key term ranking


In [10]:
def sparsity(matrix):
    total = matrix.shape[0] * matrix.shape[1]
    zeros = total - matrix.nnz
    return round((zeros / total) * 100, 2)

print("Sparsity Analysis")
print(f"OHE sparsity:    {sparsity(X_ohe)}%")
print(f"BoW sparsity:    {sparsity(X_bow)}%")
print(f"TF-IDF sparsity: {sparsity(X_tfidf)}%")

Sparsity Analysis
OHE sparsity:    98.67%
BoW sparsity:    98.67%
TF-IDF sparsity: 98.67%


In [11]:
y = df['label'].values

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print("=" * 50)
print("BoW + Logistic Regression")
print("=" * 50)
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)
print(classification_report(y_test, lr_bow.predict(X_test_bow)))

print("=" * 50)
print("BoW + Naive Bayes")
print("=" * 50)
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
print(classification_report(y_test, nb_bow.predict(X_test_bow)))

print("=" * 50)
print("TF-IDF + Logistic Regression")
print("=" * 50)
lr_tfidf = LogisticRegression()
lr_tfidf.fit(X_train_tfidf, y_train)
print(classification_report(y_test, lr_tfidf.predict(X_test_tfidf)))

print("=" * 50)
print("TF-IDF + Naive Bayes")
print("=" * 50)
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
print(classification_report(y_test, nb_tfidf.predict(X_test_tfidf)))

BoW + Logistic Regression
              precision    recall  f1-score   support

           0       0.80      0.75      0.77        16
           1       0.84      0.88      0.86        24

    accuracy                           0.82        40
   macro avg       0.82      0.81      0.82        40
weighted avg       0.82      0.82      0.82        40

BoW + Naive Bayes
              precision    recall  f1-score   support

           0       0.77      0.62      0.69        16
           1       0.78      0.88      0.82        24

    accuracy                           0.78        40
   macro avg       0.77      0.75      0.76        40
weighted avg       0.77      0.78      0.77        40

TF-IDF + Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.31      0.48        16
           1       0.69      1.00      0.81        24

    accuracy                           0.72        40
   macro avg       0.84      0.66      0.64        40
w

In [12]:
results = pd.DataFrame({
    'Method': [
        'BoW + Logistic Regression',
        'BoW + Naive Bayes',
        'TF-IDF + Logistic Regression',
        'TF-IDF + Naive Bayes'
    ],
    'Accuracy': [
        lr_bow.score(X_test_bow, y_test),
        nb_bow.score(X_test_bow, y_test),
        lr_tfidf.score(X_test_tfidf, y_test),
        nb_tfidf.score(X_test_tfidf, y_test)
    ]
})

results['Accuracy'] = results['Accuracy'].apply(lambda x: f"{round(x*100, 2)}%")
print(results.to_string(index=False))

                      Method Accuracy
   BoW + Logistic Regression    82.5%
           BoW + Naive Bayes    77.5%
TF-IDF + Logistic Regression    72.5%
        TF-IDF + Naive Bayes    67.5%


## Conclusions and Observations

### Dataset
- 200 Angry Birds app reviews
- 2 classes: Positive (1) and Negative (0)
- Vocabulary size: 1070 unique words
- Sparsity: 98.67% zeros across all matrices

### Key Findings
- BoW + Logistic Regression performed best at 82.5%
- TF-IDF underperformed because dataset is domain specific
- Words like 'game' and 'bird' appear in every review
- TF-IDF penalized these words but they were actually useful

### Real World Insight
- TF-IDF works better on large diverse datasets
- BoW can outperform TF-IDF on small domain specific datasets
- Sparsity of 98.67% shows why deep learning embeddings
  like Word2Vec are preferred at scale

### Limitations
- Dataset too small (200 reviews)
- All reviews from same app - low diversity
- No handling of emojis or slang in preprocessing